# Food Delivery Business Intelligence & Operations Analytics Platform

---

## Project Introduction

This notebook presents a structured Business Intelligence analysis of a food delivery platform's operational and transactional data. The analysis is designed to surface actionable insights across revenue performance, customer behavior, restaurant operations, and delivery efficiency.

---

## Business Problem

Food delivery platforms operate on thin margins with complex multi-sided dynamics: restaurants, customers, riders, and the platform itself all have competing incentives. Without systematic analysis, revenue leakage through discounts, cancellations, and operational inefficiency goes undetected until it becomes a structural problem.

---

## Objectives

1. Quantify the platform's financial health — revenue, AOV, discount erosion
2. Identify operational bottlenecks — KPT, rider wait, cancellations
3. Profile restaurant performance — revenue, ratings, rejection behavior
4. Understand customer patterns — retention, LTV, geography
5. Deliver prioritized, evidence-based recommendations to stakeholders

---

## Stakeholders

| Stakeholder | Primary Interest |
|---|---|
| **CEO / Business Head** | Revenue, growth, market share |
| **Operations Manager** | Delivery speed, cancellations, rider efficiency |
| **Restaurant Partnerships** | Restaurant performance, rejection rates, compliance |
| **Marketing** | Customer segments, discount ROI, AOV |
| **Finance** | Margin, discount cost, compensation/penalty P&L |

---

## Dataset Overview

- **Source:** Cleaned transactional dataset from the food delivery platform
- **Grain:** One row per order
- **Key Dimensions:** Restaurant, Customer, City, Subzone, Time
- **Key Metrics:** Revenue, Discounts, KPT, Rider Wait, Rating, Cancellations

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Plot style
sns.set_theme(style="whitegrid")

# Load data
df = pd.read_csv("../data/food_delivery_cleaned.csv")

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Dataset loaded: 21,321 rows × 35 columns


## Section 1 — Data Understanding

### 1.1 Dataset Shape & Column Index

In [23]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\nData Types")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col} = {df[col].dtypes}")


Rows: 21321
Columns: 35

Data Types
1. restaurant_id = int64
2. restaurant_name = object
3. subzone = object
4. city = object
5. order_id = int64
6. order_placed_at = object
7. order_status = object
8. delivery = object
9. distance = float64
10. items_in_order = object
11. instructions = object
12. discount_construct = object
13. bill_subtotal = float64
14. packaging_charges = float64
15. restaurant_discount_promo = float64
16. restaurant_discount_flat_offs,_freebies_&_others = float64
17. gold_discount = float64
18. brand_pack_discount = float64
19. total = float64
20. rating = float64
21. review = object
22. cancellation___rejection_reason = object
23. restaurant_compensation_cancellation = float64
24. restaurant_penalty_rejection = float64
25. kpt_duration_minutes = float64
26. rider_wait_time_minutes = float64
27. order_ready_marked = object
28. customer_complaint_tag = object
29. customer_id = object
30. hour = int64
31. day = int64
32. year = int64
33. month = int64
34. month_nam

### 1.2 Data Types

In [43]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
summary=df[numeric_cols].describe().T

summary['median'] = df[numeric_cols].median()
summary['skew'] = df[numeric_cols].skew().round(3)
summary


,count,mean,std,min,25%,50%,75%,max,median,skew
restaurant_id,21321.0,2.074413e+07,2.447193e+05,2.032061e+07,2.063570e+07,2.065987e+07,2.088265e+07,2.152306e+07,2.065987e+07,0.543
order_id,21321.0,6.354622e+09,1.230263e+08,6.086767e+09,6.250751e+09,6.357715e+09,6.456827e+09,6.573392e+09,6.357715e+09,-0.016
distance,21321.0,4.170555e+00,2.990990e+00,6.000000e-01,2.000000e+00,3.000000e+00,6.000000e+00,2.100000e+01,3.000000e+00,1.352
bill_subtotal,21321.0,7.500768e+02,4.987594e+02,5.000000e+01,4.590000e+02,6.290000e+02,8.990000e+02,1.608000e+04,6.290000e+02,5.941
packaging_charges,21321.0,3.256459e+01,2.223590e+01,0.000000e+00,1.845000e+01,2.845000e+01,3.995000e+01,6.030000e+02,2.845000e+01,4.473
restaurant_discount_promo,21321.0,6.509182e+01,8.540160e+01,0.000000e+00,0.000000e+00,8.000000e+01,1.000000e+02,4.020000e+03,8.000000e+01,7.911
"restaurant_discount_flat_offs,_freebies_&_others",21321.0,3.179506e+01,1.314871e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,7.787000e+03,0.000000e+00,13.962
gold_discount,21321.0,9.912762e-02,3.264261e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.801000e+02,0.000000e+00,49.471
brand_pack_discount,21321.0,3.039324e+00,1.707078e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.548000e+02,0.000000e+00,8.424
total,21321.0,6.826161e+02,4.653140e+02,5.250000e+01,3.874500e+02,5.974500e+02,8.379000e+02,1.266300e+04,5.974500e+02,4.500


In [ ]:
# ensure datetime dtype then print range
df['order_placed_at'] = pd.to_datetime(df['order_placed_at'], errors='coerce')
date_min = df['order_placed_at'].min().date()
date_max = df['order_placed_at'].max().date()
print(f"date range: {date_min} to {date_max} ({(date_max - date_min).days} days)")

AttributeError: 'str' object has no attribute 'date'

In [46]:
df["order_placed_at"].unique

<bound method Series.unique of 0        2024-09-10 23:38:00
1        2024-09-10 23:34:00
2        2024-09-10 15:52:00
3        2024-09-10 15:45:00
4        2024-09-10 15:04:00
                ...         
21316    2025-01-30 03:26:00
21317    2025-01-29 02:44:00
21318    2025-01-24 22:05:00
21319    2025-01-21 14:27:00
21320    2025-01-21 02:55:00
Name: order_placed_at, Length: 21321, dtype: object>